<a href="https://colab.research.google.com/github/tomaszwienke-lgtm/learning-git-task/blob/master/Zadanie_ceny_akcji.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Przesłane pliki:", list(uploaded.keys()))

Saving ca_c_f_d.csv to ca_c_f_d.csv
Saving kgh_d.csv to kgh_d.csv
Przesłane pliki: ['ca_c_f_d.csv', 'kgh_d.csv']


# Analiza cen akcji KGHM a cen miedzi (2015–2020)

Celem projektu jest zbadanie zależności między notowaniami akcji KGHM (spółki miedziowej) a ceną surowca – miedzi. Wykorzystano dane historyczne w okresie od 2015 do 2020 roku.

W ramach analizy:
- sprawdzimy jakość danych (braki, wartości odstające, duplikaty),
- połączymy oba zbiory według dat,
- zwizualizujemy ceny zamknięcia na dwóch wykresach liniowych (jeden pod drugim),
- dodamy tabelę z zestawieniem cen w poszczególnych dniach.

**Uwaga:** Ceny KGHM podane są w polskich złotych (PLN), a ceny miedzi w dolarach amerykańskich (USD). Wykresy mają osobne osie Y, więc różne jednostki nie zakłócają porównania trendów – skupiamy się na kierunku zmian.

## 1. Import bibliotek

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Opcjonalnie – aby wyświetlić wszystkie kolumny w DataFrame
pd.set_option('display.max_columns', None)

## 2. Wczytanie danych

In [ ]:
kgh = pd.read_csv('kgh_d.csv', parse_dates=['Data'])
copper = pd.read_csv('ca_c_f_d.csv', parse_dates=['Data'])

print("KGHM – pierwsze 5 wierszy:")
display(kgh.head())

print("\nMiedź – pierwsze 5 wierszy:")
display(copper.head())

KGHM – pierwsze 5 wierszy:


,Data,Otwarcie,Najwyzszy,Najnizszy,Zamkniecie,Wolumen
0,2015-01-02,102.45,102.83,101.97,102.40,309987
1,2015-01-05,102.16,102.88,99.91,100.09,479228
2,2015-01-07,100.53,105.45,99.67,104.89,966372
3,2015-01-08,105.35,107.15,105.35,107.11,711805
4,2015-01-09,107.15,107.15,103.87,104.33,563221



Miedź – pierwsze 5 wierszy:


,Data,Otwarcie,Najwyzszy,Najnizszy,Zamkniecie
0,2015-01-02,6309.0,6309.0,6309.0,6309.0
1,2015-01-05,6216.0,6216.0,6216.0,6216.0
2,2015-01-06,6191.0,6191.0,6191.0,6191.0
3,2015-01-07,6170.0,6170.0,6170.0,6170.0
4,2015-01-08,6230.5,6230.5,6230.5,6230.5


## 3. Walidacja danych

Przed wizualizacją upewniamy się, że dane są kompletne i poprawne. Sprawdzamy:
- Brakujące wartości,
- Ujemne lub zerowe ceny (niedopuszczalne),
- Duplikaty dat,
- Zgodność zakresów czasowych,
- Podstawowe statystyki opisowe.

In [ ]:
# Brakujące wartości
print("Brakujące w KGHM:\n", kgh.isnull().sum())
print("\nBrakujące w miedzi:\n", copper.isnull().sum())

# Ujemne i zerowe ceny
print("\nUjemne ceny KGHM:", (kgh['Zamkniecie'] < 0).sum())
print("Zerowe ceny KGHM:", (kgh['Zamkniecie'] == 0).sum())
print("Ujemne ceny miedzi:", (copper['Zamkniecie'] < 0).sum())
print("Zerowe ceny miedzi:", (copper['Zamkniecie'] == 0).sum())

# Duplikaty dat
print("\nDuplikaty dat KGHM:", kgh['Data'].duplicated().sum())
print("Duplikaty dat miedzi:", copper['Data'].duplicated().sum())

# Zakresy dat
print("\nZakres KGHM:", kgh['Data'].min(), "–", kgh['Data'].max())
print("Zakres miedzi:", copper['Data'].min(), "–", copper['Data'].max())

# Liczba dni
print("\nLiczba dni KGHM:", len(kgh))
print("Liczba dni miedzi:", len(copper))

# Sprawdzenie ciągłości dat (różnice między kolejnymi dniami)
kgh['Różnica_dni'] = kgh['Data'].diff().dt.days
print("\nRozkład przerw w danych KGHM:")
print(kgh['Różnica_dni'].value_counts())

# Podstawowe statystyki
print("\nStatystyki ceny zamknięcia KGHM (PLN):")
display(kgh['Zamkniecie'].describe())

print("\nStatystyki ceny zamknięcia miedzi (USD):")
display(copper['Zamkniecie'].describe())

Brakujące w KGHM:
 Data          0
Otwarcie      0
Najwyzszy     0
Najnizszy     0
Zamkniecie    0
Wolumen       0
dtype: int64

Brakujące w miedzi:
 Data          0
Otwarcie      0
Najwyzszy     0
Najnizszy     0
Zamkniecie    0
dtype: int64

Ujemne ceny KGHM: 0
Zerowe ceny KGHM: 0
Ujemne ceny miedzi: 0
Zerowe ceny miedzi: 0

Duplikaty dat KGHM: 0
Duplikaty dat miedzi: 0

Zakres KGHM: 2015-01-02 00:00:00 – 2020-06-30 00:00:00
Zakres miedzi: 2015-01-02 00:00:00 – 2020-07-13 00:00:00

Liczba dni KGHM: 1371
Liczba dni miedzi: 1400

Rozkład przerw w danych KGHM:
Różnica_dni
1.0    1061
3.0     264
2.0      20
4.0      13
5.0      11
6.0       1
Name: count, dtype: int64

Statystyki ceny zamknięcia KGHM (PLN):


,Zamkniecie
count,1371.000000
mean,92.803761
std,18.604582
min,49.400000
25%,78.836000
50%,92.480000
75%,107.820000
max,134.310000



Statystyki ceny zamknięcia miedzi (USD):


,Zamkniecie
count,1400.000000
mean,5782.307857
std,696.671266
min,4310.500000
25%,5239.875000
50%,5835.000000
75%,6214.250000
max,7262.500000


## 4. Łączenie danych

Ponieważ interesują nas dni, w których dostępne są notowania zarówno KGHM, jak i miedzi, wykonujemy `inner join` na kolumnie `Data`. Dzięki temu oba wykresy będą miały identyczną oś czasu.

In [ ]:
df = pd.merge(kgh[['Data', 'Zamkniecie']],
              copper[['Data', 'Zamkniecie']],
              on='Data',
              how='inner',
              suffixes=('_KGHM', '_Miedz'))

df.sort_values('Data', inplace=True)
df.reset_index(drop=True, inplace=True)

print("Liczba wspólnych dni:", len(df))
print("Procent wspólnych dni względem KGHM:", len(df)/len(kgh)*100, "%")
print("Procent wspólnych dni względem miedzi:", len(df)/len(copper)*100, "%")
display(df.head())

Liczba wspólnych dni: 1353
Procent wspólnych dni względem KGHM: 98.6870897155361 %
Procent wspólnych dni względem miedzi: 96.64285714285714 %


,Data,Zamkniecie_KGHM,Zamkniecie_Miedz
0,2015-01-02,102.40,6309.0
1,2015-01-05,100.09,6216.0
2,2015-01-07,104.89,6170.0
3,2015-01-08,107.11,6230.5
4,2015-01-09,104.33,6151.0


## 5. Wizualizacja

Przygotujemy trzy elementy ułożone w pionie:
- **Górny wykres** – ceny zamknięcia KGHM (linia niebieska, jednostka PLN),
- **Środkowy wykres** – ceny zamknięcia miedzi (linia pomarańczowa, jednostka USD),
- **Dolny panel** – tabela z zestawieniem cen dla każdego dnia.

Wykresy mają wspólną oś X (daty), co ułatwia porównanie trendów. Dzięki interaktywności możliwe jest przybliżanie interesujących okresów oraz odczytanie wartości po najechaniu myszką.

In [ ]:
# Stworzenie subplotów z określeniem typów
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.4, 0.4, 0.2],
    subplot_titles=('Cena zamknięcia KGHM (PLN)', 'Cena zamknięcia miedzi (USD)', 'Tabela porównawcza'),
    specs=[
        [{'type': 'xy'}],   # pierwszy wiersz – wykres
        [{'type': 'xy'}],   # drugi wiersz – wykres
        [{'type': 'table'}] # trzeci wiersz – tabela
    ]
)

# Wykres KGHM
fig.add_trace(
    go.Scatter(
        x=df['Data'],
        y=df['Zamkniecie_KGHM'],
        mode='lines',
        name='KGHM (PLN)',
        line=dict(color='blue', width=2)
    ),
    row=1, col=1
)

# Wykres miedzi
fig.add_trace(
    go.Scatter(
        x=df['Data'],
        y=df['Zamkniecie_Miedz'],
        mode='lines',
        name='Miedź (USD)',
        line=dict(color='orange', width=2)
    ),
    row=2, col=1
)

# Tabela (wszystkie wiersze)
fig.add_trace(
    go.Table(
        header=dict(
            values=['Data', 'KGHM (PLN)', 'Miedź (USD)'],
            fill_color='lightgrey',
            align='center',
            font=dict(size=12, color='black')
        ),
        cells=dict(
            values=[
                df['Data'].dt.strftime('%Y-%m-%d'),
                df['Zamkniecie_KGHM'].round(2),
                df['Zamkniecie_Miedz'].round(2)
            ],
            align='center',
            font=dict(size=11)
        )
    ),
    row=3, col=1
)

# Ukrycie osi dla tabeli (są zbędne)
fig.update_xaxes(showticklabels=False, row=3, col=1)
fig.update_yaxes(showticklabels=False, row=3, col=1)

# Formatowanie layoutu
fig.update_layout(
    title='Ceny akcji KGHM a ceny miedzi (2015–2020)',
    height=900,
    hovermode='x unified',
    showlegend=False
)

# Opisy osi
fig.update_yaxes(title_text='Cena (PLN)', row=1, col=1)
fig.update_yaxes(title_text='Cena (USD)', row=2, col=1)
fig.update_xaxes(title_text='Data', row=2, col=1)

# Wyświetlenie
fig.show()

## 6. Wnioski

- Dane są kompletne i nie zawierają błędów (brak ujemnych/zerowych cen, brak duplikatów dat).
- Ceny akcji KGHM poruszają się w przedziale 49–134 PLN, ceny miedzi 4310–7262 USD.
- Wykresy pokazują wyraźną korelację – spadki i wzrosty cen miedzi przekładają się na podobne zmiany notowań KGHM.
- Dzięki interaktywnym wykresom można w łatwy sposób badać szczegółowe okresy (np. załamanie na początku 2020 roku związane z pandemią COVID-19).

**Ważne:** Ceny KGHM i miedzi wyrażone są w różnych walutach (PLN vs USD). Ponieważ wykresy mają osobne osie Y, porównujemy wyłącznie kierunek zmian (trend), a nie wartości absolutne. W przypadku dalszej analizy (np. korelacji ilościowej) należałoby dane przeliczyć na wspólną walutę lub znormalizować.

Zastosowane podejście (walidacja danych + interaktywna wizualizacja) może być podstawą do dalszych analiz, takich jak obliczanie współczynnika korelacji, budowa modeli predykcyjnych czy tworzenie dashboardów.